# Day 3: Split by actor, baseline model, and the leakage experiment
Needs: data/metadata.csv (Day 1) and features/X_mfcc.npy (Day 2).
Only ONE thing to edit: the BASE address in the first code cell.

In [ ]:
# 1. THE ONLY ADDRESS YOU NEED (same as Day 2)
BASE = r"C:\Users\YourName\emotion-recognition"     # <-- change this to your folder

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, f1_score

BASE = Path(BASE)
(BASE / "models").mkdir(exist_ok=True)

df = pd.read_csv(BASE / "data" / "metadata.csv")
X = np.load(BASE / "features" / "X_mfcc.npy")
print("X:", X.shape, "| metadata rows:", len(df))
assert len(df) == len(X), "X and metadata do not match. Re-run Day 2."

In [ ]:
# 2. Safety check: row i of X must match row i of the metadata (re-extract 3 random clips)
SR, DURATION, OFFSET, N_MFCC, HOP_LENGTH, MAX_FRAMES = 22050, 3, 0.5, 40, 512, 130

def extract_mfcc(path):
    y, _ = librosa.load(path, sr=SR, duration=DURATION, offset=OFFSET)
    m = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=N_MFCC, hop_length=HOP_LENGTH)
    if m.shape[1] < MAX_FRAMES:
        m = np.pad(m, ((0, 0), (0, MAX_FRAMES - m.shape[1])))
    return m[:, :MAX_FRAMES].astype(np.float32)

lookup = {f.name: str(f) for f in (BASE / "data" / "ravdess").rglob("*.wav")}
df["path"] = df["path"].map(lambda p: lookup.get(Path(p).name))

rng = np.random.default_rng(42)
for i in rng.choice(len(df), size=3, replace=False):
    assert np.allclose(X[i], extract_mfcc(df.loc[i, "path"]), atol=1e-3), \
        f"Row {i} does not match its file. Re-run Day 2 so X and metadata line up."
print("Alignment OK")

In [ ]:
# 3. Encode the labels and make the split BY ACTOR
le = LabelEncoder()
y = le.fit_transform(df["emotion"])
classes = list(le.classes_)
np.save(BASE / "features" / "y.npy", y)
json.dump(classes, open(BASE / "models" / "classes.json", "w"))

tr = (df["actor"] <= 16).values          # actors 1-16  -> train
va = df["actor"].between(17, 20).values  # actors 17-20 -> validation
te = (df["actor"] >= 21).values          # actors 21-24 -> test (used only ONCE, on Day 6)

print("Classes:", classes)
print("Train / validation / test clips:", tr.sum(), va.sum(), te.sum())   # 960 / 240 / 240

In [ ]:
# 4. Helper that stores every result in results.csv (later days add more rows)
RESULTS = BASE / "results.csv"

def save_result(name, acc, f1):
    row = pd.DataFrame([{"model": name, "val_accuracy": round(float(acc), 4), "val_macro_f1": round(float(f1), 4)}])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        old = old[old["model"] != name]
        row = pd.concat([old, row], ignore_index=True)
    row.to_csv(RESULTS, index=False)
    print(row.to_string(index=False))

In [ ]:
# 5. Baseline: SVM on the mean and std of each MFCC coefficient (80 numbers per clip)
F = np.concatenate([X.mean(axis=2), X.std(axis=2)], axis=1)        # (1440, 80)

svm = Pipeline([("scaler", StandardScaler()),
                ("model", SVC(C=10, class_weight="balanced"))])
svm.fit(F[tr], y[tr])
pred = svm.predict(F[va])

acc = accuracy_score(y[va], pred)
f1 = f1_score(y[va], pred, average="macro")
print(f"Validation accuracy: {acc:.3f} | macro-F1: {f1:.3f} | chance level: {1/8:.3f}")
print(classification_report(y[va], pred, target_names=classes, zero_division=0))

In [ ]:
# 6. Confusion matrix (rows = true emotion, columns = predicted emotion)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y[va], pred, display_labels=classes,
                                        xticks_rotation=45, cmap="Blues", ax=ax)
plt.title("Baseline SVM - validation set")
plt.tight_layout()
plt.show()

In [ ]:
# 7. The leakage experiment: a RANDOM split (wrong way) vs the actor split (right way)
a, b, ya, yb = train_test_split(F, y, test_size=0.2, stratify=y, random_state=42)
svm_random = Pipeline([("scaler", StandardScaler()), ("model", SVC(C=10, class_weight="balanced"))])
svm_random.fit(a, ya)
random_acc = svm_random.score(b, yb)

print(f"Random split accuracy : {random_acc:.3f}   <- inflated (same speakers in train and test)")
print(f"Actor split accuracy  : {acc:.3f}   <- honest (new speakers)")
# Write in your own words WHY the random split is higher.

In [ ]:
# 8. Save the baseline result
save_result("Baseline SVM", acc, f1)